In [ ]:
"""[Kaggle L4] Aceptacion con DESCUENTO DE PICOS en el DFS + vectores por celda completos.

Anatomia medida (kernel dfs-budget, 22-ago): la NLL de la GT es una llanura en ~0 con 2-5
picos -- la mediana concentra el 50% en 2 celdas y el 82% en las 5 peores (L mediana 260).
Por eso el presupuesto LINEAL por posicion fue nulo (3->3 resueltas, +56% decode, barrido de
pendiente plano): las verdades no fallan por acumulacion con la longitud, fallan en picos
tempranos que agotan el presupuesto fijo de golpe.

Regla nueva: aceptar un camino si (NLL acumulada - suma de sus K=3 peores celdas) < 1.609,
con DOS guardas: cada paso individual < CAP=2.0 nats (sin tope, cualquier token entra
mientras queden exenciones y la busqueda explota), y el descuento solo perdona picos, no
basura uniforme. La cota descontada es MONOTONA no-decreciente al extender el camino (el
nuevo paso o entra al top-K expulsando al menor, o suma a la base), asi que la poda del DFS
sobrevive intacta.

Ademas guarda el VECTOR POR CELDA COMPLETO de cada GT: cualquier regla futura de aceptacion
se evalua offline gratis, sin volver a pagar GPU.

Prediccion pre-registrada: admisibilidad estructural de la GT >= 12/40 (vs 3 vieja, 7
lineal); en busqueda, el riesgo es el coste (las exenciones abren ramas en las celdas de
decision) -- si el decode por tarea excede ~2x el baseline, la regla necesita K o CAP mas
estrictos, no mas presupuesto.
"""
import glob
import json
import os
import time

import numpy as np
import torch
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import AutoModelForCausalLM

T00 = time.time()
_SDPA = [SDPBackend.EFFICIENT_ATTENTION, SDPBackend.FLASH_ATTENTION, SDPBackend.MATH]
IM_START, USER, ASSIST, NL, EOS = 14, 11, 12, 10, 15
ARC_TOKENS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]
_TOKID = torch.tensor(ARC_TOKENS, dtype=torch.long, device="cuda:0")
THR0 = -float(np.log(0.2))
K_PICOS = 3
CAP = 2.0

_COMP = None
for c in glob.glob("/kaggle/input/**/arc-agi_evaluation_challenges.json", recursive=True):
    _COMP = os.path.dirname(c)
    break
assert _COMP, "no se encontro la data de la competencia"
MDIR = None
for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
    d = os.path.dirname(c)
    if glob.glob(os.path.join(d, "*.safetensors")):
        MDIR = d
        break
assert MDIR, "no se encontro el modelo adjunto"
print(f"[setup] comp={_COMP} modelo={MDIR}", flush=True)

CHAL = json.load(open(f"{_COMP}/arc-agi_evaluation_challenges.json"))
SOLS = json.load(open(f"{_COMP}/arc-agi_evaluation_solutions.json"))

KEYS = ["e8686506", "20270e3b", "7b5033c1", "b0039139", "78332cb0", "28a6681f", "d35bdbdc",
        "dbff022c", "4c3d4a41", "446ef5d2", "dd6b8c4b", "3dc255db", "16de56c4", "97d7923e",
        "2b83f449", "bf45cf4b", "faa9f03d", "eee78d87", "31f7f899", "5dbc8537", "291dc1e1",
        "f931b4a8", "f560132c", "271d71e2", "247ef758", "7666fa5d", "21897d95", "1818057f",
        "5545f144", "135a2760", "1ae2feb7", "38007db0", "3a25b0d8", "581f7754", "6e453dd6",
        "65b59efc", "b9e38dc0", "53fb4810", "7ed72f31", "c7f57c3e"]
GATE = "38007db0"


def grid_tokens(g):
    out = []
    for i, row in enumerate(g):
        out.extend(int(c) for c in row)
        if i < len(g) - 1:
            out.append(NL)
    return out


def pair_tokens(inp, out):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + \
           [IM_START, ASSIST, NL] + grid_tokens(out) + [EOS]


def query_tokens(inp):
    return [IM_START, USER, NL] + grid_tokens(inp) + [EOS] + [IM_START, ASSIST, NL]


MODEL = AutoModelForCausalLM.from_pretrained(MDIR, torch_dtype=torch.bfloat16,
                                             device_map="cuda:0", attn_implementation="sdpa")
MODEL.eval()
MODEL.config.use_cache = True
print(f"[setup] 4B bf16 | mem={torch.cuda.memory_allocated()/1e9:.1f}GB "
      f"({time.time()-T00:.0f}s)", flush=True)


@torch.no_grad()
def prefill(toks, chunk=512):
    cache, out = None, None
    for i in range(0, len(toks), chunk):
        with sdpa_kernel(_SDPA):
            out = MODEL(input_ids=torch.tensor([toks[i:i + chunk]], device="cuda:0",
                                               dtype=torch.long),
                        past_key_values=cache, use_cache=True, return_dict=True)
        cache = out.past_key_values
    return out.logits[:, -1], cache


@torch.no_grad()
def nll_vector(prompt, ans):
    last, cache = prefill(prompt)
    parts = [last.float()]
    if len(ans) > 1:
        with sdpa_kernel(_SDPA):
            o = MODEL(input_ids=torch.tensor([ans[:-1]], device="cuda:0", dtype=torch.long),
                      past_key_values=cache, use_cache=True, return_dict=True)
        parts.append(o.logits[0].float())
    lg = torch.cat([x.view(-1, x.shape[-1]) for x in parts], 0)
    v = -torch.log_softmax(lg, -1).gather(
        1, torch.tensor(ans, device="cuda:0").unsqueeze(1)).squeeze(1)
    out = [round(x, 5) for x in v.tolist()]
    del parts, lg, cache
    torch.cuda.empty_cache()
    return out


def toks_to_grid(toks):
    rows, row = [], []
    for t in toks:
        if t == NL:
            rows.append(row)
            row = []
        elif 0 <= t <= 9:
            row.append(t)
        else:
            return None
    if row:
        rows.append(row)
    if not rows or not rows[0] or any(len(r) != len(rows[0]) for r in rows):
        return None
    return rows


def base_descontada(pasos):
    """suma - top K_PICOS. La usa tanto el check estructural como la busqueda."""
    if len(pasos) <= K_PICOS:
        return 0.0
    s = sorted(pasos, reverse=True)
    return float(sum(pasos) - sum(s[:K_PICOS]))


def admisible_picos(v):
    """La GT cabe si en TODO prefijo la base descontada queda bajo THR0 y ningun paso
    supera CAP. Chequeo incremental identico al que hara la busqueda."""
    pasos = []
    top = []
    base = 0.0
    for x in v:
        if x >= CAP:
            return False
        pasos.append(x)
        if len(top) < K_PICOS:
            top.append(x)
            top.sort()
        elif x > top[0]:
            base += top[0]
            top[0] = x
            top.sort()
        else:
            base += x
        if base >= THR0:
            return False
    return True


@torch.no_grad()
def dfs_picos(prompt, depth, budget=60):
    """DFS de produccion con la cota descontada. En cada nodo se lleva (base, top) y el
    candidato se poda si su paso >= CAP o si la base tras absorberlo >= THR0."""
    t0 = time.time()
    last, cache0 = prefill(prompt)
    leaves, path = [], []

    def rec(logits, base, top, cache, left):
        if time.time() - t0 > budget or left <= 1:
            return
        lf = logits.float()
        paso = (torch.logsumexp(lf, dim=-1, keepdim=True)
                - lf.index_select(-1, _TOKID)).cpu()
        cands = []
        for j, t in enumerate(ARC_TOKENS):
            x = paso[0, j].item()
            if x >= CAP:
                continue
            if len(top) < K_PICOS:
                nb, nt = base, sorted(top + [x])
            elif x > top[0]:
                nb, nt = base + top[0], sorted([x] + top[1:])
            else:
                nb, nt = base + x, top
            if nb < THR0:
                if t == EOS:
                    leaves.append((nb + sum(nt), list(path)))
                else:
                    cands.append((x, nb, nt, t))
        for x, nb, nt, t in sorted(cands, key=lambda c: c[0]):
            if time.time() - t0 > budget:
                return
            with sdpa_kernel(_SDPA):
                o = MODEL(input_ids=torch.tensor([[t]], device="cuda:0", dtype=torch.long),
                          past_key_values=cache, use_cache=True, return_dict=True)
            path.append(t)
            rec(o.logits[:, -1], nb, nt, o.past_key_values, left - 1)
            path.pop()

    rec(last, 0.0, [], cache0, depth)
    return leaves, time.time() - t0


def make_prompt(k):
    t = CHAL[k]
    p = []
    for x in t["train"]:
        p += pair_tokens(x["input"], x["output"])
    p += query_tokens(t["test"][0]["input"])
    return p


_v = nll_vector(make_prompt(GATE), grid_tokens(SOLS[GATE][0]) + [EOS])
print(f"[gate] {GATE}: NLL de la verdad {sum(_v):.4f}", flush=True)
assert sum(_v) < 1.0, f"gate FALLA ({sum(_v):.3f}): modelo mal cargado, se aborta"

RES = {"tareas": [], "k_picos": K_PICOS, "cap": CAP, "presupuesto_s": 60}
for i, k in enumerate(KEYS):
    prompt = make_prompt(k)
    gt = SOLS[k][0]
    gt_toks = grid_tokens(gt) + [EOS]

    v = nll_vector(prompt, gt_toks)
    adm = admisible_picos(v)
    cum = float(np.sum(v))

    try:
        hojas, secs = dfs_picos(prompt, min(931, len(gt_toks) + 40))
    except torch.cuda.OutOfMemoryError:
        print(f"[{i+1}/{len(KEYS)}] {k}: OOM en busqueda", flush=True)
        torch.cuda.empty_cache()
        hojas, secs = [], -1.0
    hojas.sort(key=lambda x: x[0])
    grids = [(sc, toks_to_grid(p)) for sc, p in hojas]
    best_ok = bool(grids and grids[0][1] == gt)
    dos_ok = bool(any(g == gt for _, g in grids[:2]))
    alguno = bool(any(g == gt for _, g in grids))

    RES["tareas"].append({
        "task": k, "L": len(gt_toks), "gt_nll": round(cum, 4),
        "admisible_picos": adm, "vector_gt": v,
        "pool": len(grids), "best_ok": best_ok, "pass2_ok": dos_ok, "gt_en_pool": alguno,
        "best_score": round(grids[0][0], 4) if grids else None,
        "decode_s": round(secs, 1)})
    print(f"[{i+1}/{len(KEYS)}] {k}: L={len(gt_toks)} adm_picos={adm} | pool {len(grids)} | "
          f"gt_en_pool {alguno} | pass2 {dos_ok} | {secs:.0f}s", flush=True)
    json.dump(RES, open("/kaggle/working/dfs_spikes.json", "w"), indent=1)
    torch.cuda.empty_cache()

t = RES["tareas"]
resumen = {
    "n": len(t),
    "admisible_picos": sum(x["admisible_picos"] for x in t),
    "gt_en_pool_busqueda": sum(x["gt_en_pool"] for x in t),
    "pass2": sum(x["pass2_ok"] for x in t),
    "best_ok": sum(x["best_ok"] for x in t),
    "pool_total": sum(x["pool"] for x in t),
    "decode_total_s": round(sum(max(x["decode_s"], 0) for x in t), 1),
    "baselines": {"fija_1609": {"gt_en_pool": 3, "decode_s": 397},
                  "lineal_b0202": {"gt_en_pool": 3, "decode_s": 621}},
}
RES["resumen"] = resumen
json.dump(RES, open("/kaggle/working/dfs_spikes.json", "w"), indent=1)
print("RESULT_JSON:" + json.dumps(resumen), flush=True)
print(f"[fin] {time.time()-T00:.0f}s", flush=True)
